# Jour 1 · Chapitre 1 — Scénario : l'audit des données du premier jour
## Formation Beobank · SAS → Python · Corrigé formateur

**L'histoire.** Lundi, 9h. Vous rejoignez l'équipe Data de Beobank. Votre responsable, Karim,
vous transmet un dossier de 5 fichiers CSV et vous dit :
*"Avant de faire la moindre analyse, je veux savoir ce qu'on a vraiment dans ces fichiers :
combien de lignes, quelles colonnes, où sont les trous, et si les tables se parlent entre elles.
Tu me fais un état des lieux pour 11h ?"*

**Comment travailler.** Chaque cellule contient des trous `______`. Remplacez chaque trou par le
bon mot, la bonne fonction ou la bonne valeur. Les commentaires vous guident **sans donner la
réponse** : c'est à vous de retrouver la notion dans la matière.

📎 **Matière à relire :** `Dataset_Beobank.ipynb` (même dossier) — dictionnaire des colonnes,
convention de lecture, relations entre tables.

> Exécutez les cellules **dans l'ordre**. Un résultat attendu est parfois indiqué en commentaire
> (`# → ...`) pour vous auto-corriger.

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Imports et paramètres de lecture ──────────────────────────────────────
import pandas as pd                                   # pandas : la bibliothèque des tableaux de données
DATA = "../../data/"                                  # dossier partagé par tous les chapitres
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")  # export SAS : ";" séparateur, "." = valeur manquante
print("Prêt.")

---
## Mission 1 — Ouvrir les 5 fichiers

*"Commence par tout charger, on verra après."*

Chargez les 5 tables. Rappel : la même convention de lecture s'applique à **tous** les fichiers.

In [ ]:
# ── Chargement des 5 tables ───────────────────────────────────────────────
# Quelle fonction pandas transforme un fichier CSV en tableau (DataFrame) ?
ctr  = pd.read_csv(DATA + "CTR.csv", **PARAMS)        # contrats / comptes
# Quel est le nom du fichier des clients ("tiers") ?
tie  = pd.read_csv(DATA + "TIE.csv", **PARAMS)
adr  = pd.read_csv(DATA + "TIE_ADR.csv", **PARAMS)     # adresses des clients
# Oublier les paramètres ici = mauvais séparateur => 1 seule colonne illisible
lien = pd.read_csv(DATA + "TIE_X_CTR.csv", **PARAMS)  # qui possède quel compte
txn  = pd.read_csv(DATA + "TXN_X_CTR.csv", **PARAMS)   # mouvements sur les comptes
print("5 tables chargées")

---
## Mission 2 — Mesurer chaque table

*"Combien de lignes et de colonnes dans chaque fichier ? Et quels types de données ?"*

In [ ]:
# ── Dimensions ────────────────────────────────────────────────────────────
# Quel attribut renvoie le tuple (nombre_de_lignes, nombre_de_colonnes) ?
print("CTR :", ctr.shape)                 # → (200, 11)
# Quelle fonction Python de base compte les lignes d'un DataFrame ?
print("TXN :", len(txn), "mouvements")    # → 1260 mouvements
# Quel attribut donne la liste des noms de colonnes ?
print(list(ctr.columns))
# Quel attribut (sans parenthèses) donne le type de chaque colonne ?
ctr.dtypes

✍️ **Réponse attendue :** `DAT_OUV_CTR`, `DAT_ECV_CTR`, `DAT_CLO_CTR`, `DAT_MAJ_SLD` sont lues comme
du texte. On ne peut pas soustraire deux textes : il faudra les convertir en dates
(`pd.to_datetime`) avant tout calcul de durée.

---
## Mission 3 — Vérifier les identifiants

*"Un compte = une ligne dans CTR, un client = une ligne dans TIE. Tu peux me le prouver ?"*

In [ ]:
# ── Unicité des clés ──────────────────────────────────────────────────────
# Quel attribut d'une colonne répond True si aucune valeur n'est répétée ?
print("IDT_AC unique dans CTR :", ctr["IDT_AC"].is_unique)      # → True
# Quelle méthode compte le nombre de valeurs DIFFÉRENTES ?
print("Clients distincts :", tie["IDT_PI"].nunique())           # → 100

---
## Mission 4 — Où sont les trous ?

*"Le comité risque veut savoir si on peut se fier aux soldes. Quel pourcentage est manquant ?"*

In [ ]:
# ── Valeurs manquantes par colonne ────────────────────────────────────────
# isna() marque chaque case vide par True ; quelle méthode additionne ces True par colonne ?
manquants = ctr.isna().sum()
taux = (manquants / len(ctr) * 100).round(1)          # en pourcentage
# On veut les 3 colonnes LES PLUS incomplètes en premier : tri croissant ou décroissant ?
taux.sort_values(ascending=False).head(3)          # → MNT_INI, SLD_DSP, SLD_CTR

✍️ **Ce que ça veut dire pour la banque :** près de 7 soldes sur 10 sont absents (`SLD_CTR`).
Une moyenne calculée sur les seuls soldes connus ne représente pas le portefeuille : il faut
toujours annoncer le nombre de soldes réellement disponibles à côté du chiffre.

---
## Mission 5 — Comprendre les codes

*"C'est quoi ces 4 et ces 6 dans COD_ECV_CTR ? Et nos clients parlent quelle langue ?"*

In [ ]:
# ── Répartition des codes ─────────────────────────────────────────────────
# Quelle méthode compte combien de fois chaque valeur apparaît ?
print(ctr["COD_ECV_CTR"].value_counts())          # → 4 : 105, 6 : 95
# Quel paramètre de value_counts() donne des proportions au lieu de nombres ?
tie["COD_LNG_CTR"].value_counts(normalize=True)    # → FR 0.67, NL 0.33

---
## Mission 6 — Les tables se parlent-elles ?

*"Si je relie les mouvements aux contrats, je perds combien de lignes ?"*

La relation attendue : `CTR ──(IDT_AC)── TXN_X_CTR` et `TIE ──(IDT_PI)── TIE_X_CTR`.

In [ ]:
# ── Couverture des jointures ──────────────────────────────────────────────
# Quelle méthode teste, ligne par ligne, si une valeur fait partie d'une autre liste ?
connus = txn["IDT_AC"].isin(ctr["IDT_AC"])
# connus est une colonne de True/False : comment compter les True ?
print(connus.sum(), "mouvements sur", len(txn), "retrouvent leur contrat dans CTR")
# Tous les titulaires de TIE_X_CTR existent-ils dans TIE ? (quelle clé relie ces deux tables ?)
print("Titulaires tous connus :", lien["IDT_PI"].isin(tie["IDT_PI"]).all())   # → True

---
## Mission 7 — Le piège des dates SAS

*"Les dates de mise à jour d'adresse ont un format bizarre, tu peux regarder ?"*

`DAT_MAJ_ADR` ressemble à `24NOV2025` (format SAS `DATE9.` : jour, mois abrégé en anglais, année).

In [ ]:
# ── Conversion d'une date au format SAS ───────────────────────────────────
print(adr["DAT_MAJ_ADR"].head(3).tolist())            # texte brut : ['24NOV2025', ...]
# Quelle fonction pandas convertit du texte en vraie date ?
# Codes de format : %d = jour, %b = mois abrégé (NOV), %Y = année sur 4 chiffres
adr["DAT_MAJ_ADR"] = pd.to_datetime(adr["DAT_MAJ_ADR"], format="%d%b%Y")
adr["DAT_MAJ_ADR"].head(3)                            # → 2025-11-24, 2025-11-24, 2026-01-09

---
## 🏁 Mission finale — Le rapport de 11h

*"Fais-moi une fiche d'identité du dataset, en quelques lignes, que je peux coller dans un mail."*

Format attendu :
```
nb_contrats            : 200
nb_clients             : 100
pct_solde_manquant     : 68.5
devise_principale      : EUR
```

In [ ]:
# ── Fiche d'identité du dataset ───────────────────────────────────────────
bilan = {
    "nb_contrats": len(ctr),                                        # nombre de lignes de CTR
    "nb_clients": tie["IDT_PI"].nunique(),
    # La moyenne d'une colonne True/False = proportion de True
    "pct_solde_manquant": round(ctr["SLD_CTR"].isna().mean() * 100, 1),
    # Quelle méthode renvoie l'ÉTIQUETTE de la valeur la plus grande ?
    "devise_principale": ctr["COD_DEV"].value_counts().idxmax(),
}
# Quelle méthode d'un dictionnaire donne les paires (clé, valeur) ?
for cle, valeur in bilan.items():
    print(f"{cle:22} : {valeur}")

✍️ **Exemple de mail :** *"Le dataset contient 200 contrats et 100 clients. 68,5 % des soldes
sont manquants, donc toute statistique sur les soldes doit préciser sa base. Seule une petite
partie des mouvements (TXN) retrouve son contrat dans CTR : les analyses de mouvements se feront
sur TXN seul."*